In [1]:
from abc import ABC, abstractmethod
from enum import Enum
from datetime import datetime

**ENUMS**

In [2]:
class VehicleType(Enum):
    BIKE="bike"
    CAR="car"
    TRUCK="truck"

In [21]:
class SpotType(Enum):
    BIKE="bike"
    COMPACT="compact"
    LARGE="large"

**VEHICLE: Simple entity**

In [22]:
class Vehicle:
    def __init__(self,number:str,vehicle_type:VehicleType):
        self.number=number
        self.vehicle_type=vehicle_type
    def __str__(self):
        return f"{self.vehicle_type.value.upper()}-{self.number}"

**PARKING SPOT: Spot khud apni availability aur vehicle compatibility manage krega**

In [39]:
class ParkingSpot:
    def __init__(self,spot_id:int,spot_type:SpotType):
        self.spot_id=spot_id
        self.spot_type=spot_type
        self.vehicle=None
    def is_available(self):
        return self.vehicle is None
    def can_fit(self,vehicle: Vehicle):
        if vehicle.vehicle_type==VehicleType.BIKE:
            return self.spot_type==SpotType.BIKE
            
        if vehicle.vehicle_type==VehicleType.CAR:
            return self.spot_type in [
                SpotType.COMPACT,
                SpotType.LARGE
            ]
        if vehicle.vehicle_type==VehicleType.TRUCK:
            return self.spot_type==SpotType.LARGE
        return False
    def park(self,vehicle:Vehicle):
        if not self.is_available():
            return False
        if not self.can_fit(vehicle):
            return False
        if not self.is_available():
            return False
        if not self.can_fit(vehicle):
            return False
        self.vehicle=vehicle
        return True
    def remove_vehicle(self):
        vehicle=self.vehicle
        self.vehicle=None
        return vehicle

**Parking Floor: Floor apne parking spots ko manage krega aur suitable spot find krne me help krega**

In [40]:
class ParkingFloor:
    def __init__(self,floor_id: int):
        self.floor_id=floor_id
        self.spots=[]
    def add_spot(self,spot: ParkingSpot):
        self.spots.append(spot)
    def find_spot(self,vehicle:Vehicle):
        for spot in self.spots:
            if spot.is_available() and spot.can_fit(vehicle):
                return spot
        return None

**Ticket: Ticket ek parking session ko represent krega**

In [41]:
class Ticket:
    def __init__(self,ticket_id: int,vehicle: Vehicle, spot: ParkingSpot):
        self.ticket_id=ticket_id
        self.vehicle=vehicle
        self.spot=spot
        self.entry_time=datetime.now()
        self.entry_time=datetime.now()
        self.exit_time=None
    def close(self):
        self.exit_time=datetime.now()

**Pricing Strategy**

In [42]:
class PricingStrategy(ABC):
    @abstractmethod
    def calculate_price(self,ticket:Ticket):
        pass

class HourlyPricing(PricingStrategy):
    def calculate_price(self,ticket: Ticket):
        end_time=ticket.exit_time or datetime.now()
        duration=end_time-ticket.entry_time
        hours=duration.total_seconds()/3600
        hours=max(1,int(hours)+(hours%1>0))
        return hours*50

**PARKING LOT:   ParkingLot overall parking operation coordinate krega, but hm isko pricing yaa payment ki responsiblity nhi denge(SRP)**

In [61]:
class ParkingLot:
    def __init__(self,pricing_strategy:PricingStrategy):
        self.floor=[]
        self.pricing_strategy=pricing_strategy
        self.next_ticket_id=1
        self.active_tickets={}
    def add_floor(self,floor:ParkingFloor):
        self.floor.append(floor)
    def park_vehicle(self,vehicle:Vehicle):
        for floor in self.floor:
            spot=floor.find_spot(vehicle)
            if spot:
                spot.park(vehicle)
                ticket=Ticket(
                self.next_ticket_id,
                vehicle,
                spot
                )
                self.next_ticket_id+=1
                self.active_tickets[ticket.ticket_id]=ticket
                return ticket
        return None
    def exit_vehicles(self,ticket_id:int):
        ticket=self.active_tickets.get(ticket_id)
        if ticket is None:
            return None
        ticket.close()
        price=self.pricing_strategy.calculate_price(ticket)
        ticket.spot.remove_vehicle()
        del self.active_tickets[ticket_id]
        return price

In [57]:
parking_lot=ParkingLot(pricing_strategy=HourlyPricing()) #ParkingLot ko ye nhi pta ki price kaise calculate ho rh
floor1=ParkingFloor(floor_id=1)
floor1.add_spot(ParkingSpot(1,SpotType.BIKE))
floor1.add_spot(ParkingSpot(2,SpotType.COMPACT))
floor1.add_spot(ParkingSpot(3,SpotType.LARGE))
parking_lot.add_floor(floor1)

In [58]:
car=Vehicle(number="UP14AB1234",
            vehicle_type=VehicleType.CAR
)
ticket=parking_lot.park_vehicle(car)
print("Ticket ID:",ticket.ticket_id)
print("Vehicle:",ticket.vehicle)
print("Spot:",ticket.spot.spot_id)

Ticket ID: 1
Vehicle: CAR-UP14AB1234
Spot: 2


In [59]:
price=parking_lot.exit_vehicles(ticket.ticket_id)
print("Parking Price: rs",price)

Parking Price: rs 50


- Encapsulation: Classes apna state manage kr rh h
- Inheritance: HourlyPricing --> Pricing Strategy
- Polymorphism: Different Pricing Strategy
- Abstraction: Pricing Strategy
- Composition: ParkingLot-->Floors-->Spots
- StrategyPattern: PricingStrategy
- Dependency Injection: ParkingLot(pricing_strategy)
- SRP: Har class ka focused responsibility

S-SRP:

- ParkingSpot --> spot management
- Ticket --> parking session
- PricingStrategy --> pricing
- PaymentStrategy --> payment

**Each class has a focused responsibility**

O-OCP
suppose: WeekendPricing, UPIPayment and NearestSpotStrategy to be added

add kr skte h without modigying core classes

D-DIP

ParkingLot-->PricingStrategy<--HourlyPricing

ParkingLot concrete implementation pe depend nhi krta

L+I
Is design me inheritance mainly abstractions ke through controlled h, and interfaces ko small/focused rkha ja skta h

- first clarify the requirements. I've assumed multiple floors, multiple vehicle types, different parking spot types, ticket-based parking, configurable pricing and multiple payment methods.
- The core entities would be ParkingLot, ParkingFloor, ParkingSpot, Vehicle, Ticket and Payment. ParkingLot contains floors, each floor contains spots, and a vehicle gets assigned to a suitable spot with a ticket representing the parking session.
- For behaviors that are likely to change, such as spot allocation, pricing and payment, I'll use abstractions with Strategy Pattern. This avoids hardcoding these behaviors and allows new strategies to be added without modifying existing classes.
- For example, PricingStrategy can have HourlyPricing and WeekendPricing implementations, while PaymentStrategy can have UPI, Card and Cash implementations.
- The entry flow is Vehicle → ParkingLot → Spot Allocation → ParkingSpot → Ticket. The exit flow is Ticket → Pricing → Payment → Release Spot
- This design follows SRP, OCP and DIP, and keeps the system loosely coupled and extensible.